# Kidney Variant Effect Scores (Kidzoi vs Kidformer)

This notebook loads model predictions that estimate how much a single DNA variant changes **chromatin accessibility** in 10 different kidney cell types.

**Example variant used throughout this notebook:**

| chrom | pos | id | ref allele | alt allele |
|---|---|---|---|---|
| chr1 | 7,118,071 | 1:7118071 | A | G |

So we're asking: *"If the A at this exact position in the genome were changed to a G, how would that change predicted chromatin accessibility, in each of the 10 kidney cell types?"*

**Important background:** both models were **trained on kidney scATAC-seq data**. scATAC-seq measures which regions of DNA are "open" (accessible) in a given cell — open regions are generally where genes are actively being switched on. So the models predict **accessibility**, not gene expression directly.

**The two models compared here:**
- `kidzoi`
- `kidformer`

**The 10 kidney cell types:** (index 0 = Immune, index 9 = LOH, etc.):

| Index | Label |
|---|---|
| 0 | Immune |
| 1 | Str |
| 2 | Pod |
| 3 | CD |
| 4 | CFH |
| 5 | Tcell |
| 6 | End |
| 7 | DT |
| 8 | PT |
| 9 | LOH |

In [10]:
import h5py

In [28]:
kidzoi_score = h5py.File('testing_kidzoi/sad.h5', 'r')
kidformer_score = h5py.File('testing_kidformer/sad.h5', 'r')

In [29]:
kidformer_score["REF"], kidformer_score["ALT"]

(<HDF5 dataset "REF": shape (1, 8, 10), type "<f2">,
 <HDF5 dataset "ALT": shape (1, 8, 10), type "<f2">)

In [30]:
kidzoi_score["REF"], kidzoi_score["ALT"]

(<HDF5 dataset "REF": shape (1, 32, 10), type "<f2">,
 <HDF5 dataset "ALT": shape (1, 32, 10), type "<f2">)

## Understanding the `REF` and `ALT` dataset

Both models don't just output one number per cell type right away. First, they scan a window of DNA (524288 bp kidzoi and 196608 by kidformer) around the variant and predict accessibility in many small consecutive chunks called **bins**. Only the bins covering the **central 1024 base pairs (bp)** around the variant are kept for scoring — the bins further away are discarded because the effect of a variant is most reliably captured right around itself.

```
kidformer_score["REF"]
<HDF5 dataset "REF": shape (1, 8, 10), type "<f2">
kidformer_score["ALT"]
<HDF5 dataset "ALT": shape (1, 8, 10), type "<f2">

kidzoi_score["REF"]
<HDF5 dataset "REF": shape (1, 32, 10), type "<f2">

kidzoi_score["ALT"]
<HDF5 dataset "ALT": shape (1, 32, 10), type "<f2">
```

The shape is `(variant, bins, cell_types)`. Here `1` = one variant, `10` = the 10 kidney cell types. The middle number is the number of bins that fit in that central 1024 bp window, which differs between the two models because **each model uses a different bin size**:

- **kidzoi**: bin size = 32 bp → 1024 bp / 32 bp = **32 bins**
- **kidformer**: bin size = 128 bp → 1024 bp / 128 bp = **8 bins**

In other words, kidzoi looks at the DNA in finer detail (smaller, more numerous slices), while kidformer looks at it in coarser detail (bigger, fewer slices), but both are covering exactly the same 1024 bp stretch of DNA centered on the variant. `REF` specifically holds the predictions for the **original/reference sequence** (before the variant is introduced) and `ALT` dataset for the sequence *with* the variant.

In [31]:
kidzoi_score['SAD'][:]

array([[  -6.31 ,  -16.61 ,   82.   ,  -78.75 , -255.6  ,   -9.94 ,
          -5.758, -368.5  ,  -55.66 , -506.5  ]], dtype=float16)

In [26]:
kidformer_score['SAD'][:]

array([[ -19.77, -120.3 ,   10.47, -185.9 , -565.  ,  -17.94,  -18.83,
        -615.  , -304.8 , -821.5 ]], dtype=float16)

**What is a SAD score?**
"SAD" stands for *SNP Activity Difference*. Both **Kidzoi** and **Kidformer** are deep learning models that take a stretch of DNA sequence and predict how "active" that region is (roughly: how strongly nearby genes would be expressed/turned on).

To score a variant, the model is run twice on the same DNA window:
- once with the **reference** (normal) sequence
- once with the **variant** (mutated) sequence

The difference between these two predictions is the **SAD score**. A SAD score near 0 means the model thinks the variant barely changes gene activity. A large positive or negative SAD score means the model predicts the variant has a strong effect (increasing or decreasing activity).



```
kidzoi_score['SAD'][:]
array([[  -6.31 ,  -16.61 ,   82.   ,  -78.75 , -255.6  ,   -9.94 ,
          -5.758, -368.5  ,  -55.66 , -506.5  ]], dtype=float16)

kidformer_score['SAD'][:]
array([[ -19.77, -120.3 ,   10.47, -185.9 , -565.  ,  -17.94,  -18.83,
        -615.  , -304.8 , -821.5 ]], dtype=float16)
```

Shape is `(1, 10)`: one row for our one variant, one column per kidney cell type (same order as `targets_df`).

**How to read a SAD value:**
- **Negative** → the model predicts the variant (A→G) makes that DNA region **less accessible** ("closes it up") in that cell type — i.e. likely reduces gene activity there.
- **Positive** → the model predicts the variant makes that region **more accessible** ("opens it up") in that cell type — i.e. likely increases gene activity there.
- **Bigger magnitude (further from 0)** → stronger predicted effect, regardless of sign.

**Reading the actual results for chr1:7118071 A>G:**
- Both models agree the strongest effect is in **LOH (Loop of Henle)** — strongly negative in both (-506.5 for kidzoi, -821.5 for kidformer) — so both predict this variant substantially reduces accessibility specifically in Loop of Henle cells.
- The second-strongest predicted effect in both models is in **DT (Distal Tubule)**, again negative in both.
- Interestingly, the two models **agree in direction** for all **cell types**: just not in magnitude. Only for the **Pod** cell type, out of all 10, where the predicted effect is an *increase* rather than a decrease in accessibility.
- For every other cell type (Immune, Str, CD, CFH, Tcell, End, DT), both models agree the variant is predicted to **decrease** accessibility, though kidformer consistently reports larger-magnitude scores than kidzoi for the same cell type.

In [48]:
kidzoi_score.close()
kidformer_score.close()